# Fase 2: Auditoría Adversaria: Ataques Avanzados

Autor: Daniel Gomollón Embid

Trabajo Fin de Grado: Análisis, Explotación y Mitigación de Vulnerabilidades de Sistemas de Detección de intrusiones basados en Machine Learning

Fecha: 26/04/2026

**Universidad de Zaragoza**

# ETAPA 3: Simulación de Ataques Reales: SPECTRA y DLA-Digital Twin

# **SPECTRA — Transferibilidad de Ataques Adversariales (Black Box - No Brute Force)**

## Hipótesis
Un atacante real no tiene acceso al modelo víctima (BigFlow-NIDS ResNet/LightGBM).
Solo dispone de un dataset público de tráfico de red (NF-UQ-NIDS-V2) con 44 features (finalmente 30 para adaptarnos a NetFlow puro y ser realistas ante el modelo víctima) - Sin IP Behavior Buffer ni features sintéticas.

¿Cuántos ataques adversariales generados sobre ese modelo sustituto (surrogate)
consiguen evadir al modelo víctima real?

## Métrica central: ITF (Inference Transfer Factor)
ITF = ataques que evaden víctima / ataques que evaden surrogate

------------------------

## Montar Drive y Configuraciones

In [ ]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# TODO: Cambia esta ruta a la carpeta raíz de tu TFG en Drive
PROJECT_ROOT = '/content/drive/MyDrive/Codigo_TFG'
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Importar librerías core
import torch
import numpy as np
import polars as pl

import torch
import lightgbm as lgb
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, f1_score

# Importar nuestros módulos
from src.config import Config
from src.helpers import set_seed
from src.utils.domain_constraints import DomainConstraints

# Fijar semilla para reproducibilidad
set_seed(Config.SEED)

# Comprobar hardware
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[-] Ejecutando en dispositivo: {device.upper()}")

## 1. Carga del modelo víctima y datos de BigFlow-NIDS

El modelo víctima nunca se toca durante el entrenamiento del surrogate.
Solo se usa para evaluar cuántos ataques adversariales se transfieren.

In [ ]:
import joblib
from src.models.wrapper_attacks import load_resnet_for_attack, load_lgbm_for_attack

DATA_PATH   = os.path.join(PROJECT_ROOT, 'data', 'processed', 'resultados_2_buffer')
MODELS_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'models')
SURROGATE_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'resultados_3_surrogate')

# Cargar datos de test escalados
print("[-] Cargando datos de test BigFlow-NIDS...")
X_test_victim_sc = np.load(os.path.join(DATA_PATH, 'X_test.npy')).astype(np.float32)
y_test_victim    = np.load(os.path.join(DATA_PATH, 'y_test.npy'))
print(f"   [✓] X_test escalado: {X_test_victim_sc.shape}")

# Cargar modelos víctima
print("\n[-] Cargando modelos víctima...")
resnet_victim  = load_resnet_for_attack(
    device=device,
    input_dim=X_test_victim_sc.shape[1],
    models_path=MODELS_PATH
)
lgbm_victim    = load_lgbm_for_attack(models_path=MODELS_PATH)
lgbm_explainer = joblib.load(os.path.join(MODELS_PATH, 'lgbm', 'lgbm_shap_explainer.pkl'))
print("   [✓] ResNet víctima cargada")
print("   [✓] LightGBM víctima cargado")

# Motor físico — necesario para invertir el escalado
dc = DomainConstraints.from_artifacts(models_path=MODELS_PATH, data_path=DATA_PATH)

# Invertir escalado para LightGBM (entrenado con datos raw)
print("\n[-] Invirtiendo escalado para LightGBM...")
X_test_victim_raw = dc.to_physical_space(X_test_victim_sc)
print(f"   [✓] X_test raw: {X_test_victim_raw.shape}")

# Submuestra de ataques — mismos índices para ambos espacios
np.random.seed(Config.SEED)
idx_attacks = np.where(y_test_victim != 0)[0]
np.random.shuffle(idx_attacks)
idx_attacks = idx_attacks[:2500]

X_victim_attacks_sc  = X_test_victim_sc[idx_attacks]   # para ResNet
X_victim_attacks_raw = X_test_victim_raw[idx_attacks]  # para LightGBM
y_victim_attacks     = y_test_victim[idx_attacks]

print(f"\n   [✓] Muestras de ataque: {len(idx_attacks):,}")

# Baseline — detección sin perturbación
y_pred_baseline_resnet = resnet_victim.predict(X_victim_attacks_sc)
y_pred_baseline_lgbm   = lgbm_victim.predict(X_victim_attacks_raw)

detection_resnet = (y_pred_baseline_resnet != 0).mean()
detection_lgbm   = (y_pred_baseline_lgbm   != 0).mean()

print(f"\n   Detección baseline ResNet  : {detection_resnet*100:.1f}%")
print(f"   Detección baseline LightGBM: {detection_lgbm*100:.1f}%")
print(f"\n   Ambos deberían estar ~95%+ para validar el experimento")

## 2. Carga y entrenamiento del modelo surrogate

El surrogate se entrena con NF-UQ-NIDS-V2 (44 features, sin IP Buffer).
El atacante nunca ha visto BigFlow-NIDS ni el modelo víctima.

Las features BUF_* se rellenan a cero al transferir los ataques al espacio
del modelo víctima — el atacante desconoce la existencia del buffer.

In [ ]:
import time
import lightgbm as lgb
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import seaborn as sns

SURROGATE_MODELS_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'models', 'surrogate')

print("[-] Cargando datos surrogate (NF-UQ-NIDS-V2, 30 features)...")
X_surr_train = np.load(os.path.join(SURROGATE_DATA_PATH, 'X_train.npy')).astype(np.float32)
y_surr_train = np.load(os.path.join(SURROGATE_DATA_PATH, 'y_train.npy'))
X_surr_val   = np.load(os.path.join(SURROGATE_DATA_PATH, 'X_val.npy')).astype(np.float32)
y_surr_val   = np.load(os.path.join(SURROGATE_DATA_PATH, 'y_val.npy'))
X_surr_test  = np.load(os.path.join(SURROGATE_DATA_PATH, 'X_test.npy')).astype(np.float32)
y_surr_test  = np.load(os.path.join(SURROGATE_DATA_PATH, 'y_test.npy'))
feature_names_surrogate = np.load(
    os.path.join(SURROGATE_MODELS_PATH, 'feature_names.npy'), allow_pickle=True
)

print(f"   [✓] Train: {X_surr_train.shape} | Val: {X_surr_val.shape} | Test: {X_surr_test.shape}")
print(f"   [✓] Features surrogate: {len(feature_names_surrogate)}")
print(f"   [✓] Features víctima  : {len(dc.feature_names)} (66 con BUF_*)")
print(f"   Features no disponibles para el atacante: "
      f"{len(dc.feature_names) - len(feature_names_surrogate)} (BUF_* + sintéticas)")

In [ ]:
# Entrenar LightGBM surrogate
print("\n[-] Entrenando LightGBM surrogate...")
t0 = time.time()

surrogate_lgbm = lgb.LGBMClassifier(
    objective         = 'multiclass',
    num_class         = len(np.unique(y_surr_train)),
    n_estimators      = 500,
    learning_rate     = 0.05,
    num_leaves        = 63,
    min_child_samples = 20,
    colsample_bytree  = 0.8,
    subsample         = 0.8,
    n_jobs            = -1,
    random_state      = Config.SEED,
    verbose           = -1,
)

surrogate_lgbm.fit(
    X_surr_train, y_surr_train,
    eval_set  = [(X_surr_val, y_surr_val)],
    callbacks = [
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(period=-1),
    ],
)

t_train = time.time() - t0

# Predicciones sobre test
y_pred_surr = surrogate_lgbm.predict(X_surr_test)
f1_surr     = f1_score(y_surr_test, y_pred_surr, average='macro')

print(f"   [✓] Entrenado en {t_train:.1f}s | Árboles: {surrogate_lgbm.booster_.num_trees()}")
print(f"   F1-macro (test NF-UQ): {f1_surr*100:.2f}%")

# Métricas por clase
CLASS_NAMES_SURR = Config.CLASS_NAMES  # mismas 8 clases
print(f"\n   Métricas por clase (test NF-UQ-NIDS-V2):")
print(f"   {'Clase':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Soporte':>10}")
print(f"   {'─'*62}")

report = classification_report(
    y_surr_test, y_pred_surr,
    target_names=CLASS_NAMES_SURR,
    output_dict=True,
    zero_division=0,
)
for cls_name in CLASS_NAMES_SURR:
    if cls_name in report:
        r = report[cls_name]
        print(f"   {cls_name:<20} {r['precision']:>10.3f} {r['recall']:>10.3f} "
              f"{r['f1-score']:>10.3f} {int(r['support']):>10,}")

print(f"\n   Nota: El surrogate tiene 95%+ porque NF-UQ tiene patrones más limpios")
print(f"         que BigFlow-NIDS. Lo importante es la transferibilidad al víctima.")

# Guardar surrogate
os.makedirs(SURROGATE_MODELS_PATH, exist_ok=True)
joblib.dump(surrogate_lgbm, os.path.join(SURROGATE_MODELS_PATH, 'surrogate_lgbm.pkl'))
print(f"\n   [✓] Surrogate guardado en {SURROGATE_MODELS_PATH}")

## 3. Construcción del espacio de ataque

El atacante genera ataques adversariales sobre el surrogate.
Para transferirlos al modelo víctima necesita expandir el vector
de 44 features a 66 features — las BUF_* se ponen a cero
(el atacante desconoce el IP Buffer del NIDS víctima).

## **DC ligero para el surrogate**

In [ ]:
"""
Domain Constraints ligero para el surrogate NF-UQ-NIDS-V2.
El atacante conoce las restricciones físicas del protocolo de red
aunque no conozca el modelo víctima ni su pipeline de features.
"""
from dataclasses import dataclass, field
import numpy as np
from sklearn.preprocessing import QuantileTransformer

@dataclass
class SurrogateDomainConstraints:
    """
    Motor físico simplificado para el surrogate de 30 features.

    El atacante sabe que:
      - No puede generar bytes negativos
      - No puede poner puertos > 65535
      - Las features derivadas deben ser coherentes
      - El protocolo TCP tiene restricciones de tamaño de paquete

    No necesita saber nada del modelo víctima para esto.
    """
    feature_names  : np.ndarray
    scaler         : object          # QuantileTransformer del surrogate
    forward_mask   : np.ndarray = field(init=False)
    perturbable_mask: np.ndarray = field(init=False)

    # Las mismas categorías físicas que BigFlow — el protocolo no cambia
    FORWARD_SURR = [
        'IN_BYTES', 'IN_PKTS', 'MIN_TTL',
        'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT',
        'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
        'SRC_TO_DST_SECOND_BYTES', 'SRC_TO_DST_AVG_THROUGHPUT',
        'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
        'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
        'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN',
        'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX',
        'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV',
        'FLOW_DURATION_MILLISECONDS', 'DURATION_IN',
    ]

    # Límites físicos absolutos — mismos que BigFlow porque son del protocolo
    PHYSICAL_BOUNDS_SURR = {
        'IN_BYTES'                   : (40.0,    np.inf),
        'IN_PKTS'                    : (1.0,     140_225.0),
        'FLOW_DURATION_MILLISECONDS' : (0.0,     120_830.0),
        'MIN_TTL'                    : (0.0,     255.0),
        'LONGEST_FLOW_PKT'           : (28.0,    1500.0),
        'SHORTEST_FLOW_PKT'          : (28.0,    1500.0),
        'MIN_IP_PKT_LEN'             : (0.0,     1500.0),
        'MAX_IP_PKT_LEN'             : (28.0,    1500.0),
        'TCP_WIN_MAX_IN'             : (0.0,     65535.0),
        'SRC_TO_DST_IAT_MIN'         : (0.0,     59_995.0),
        'SRC_TO_DST_IAT_MAX'         : (0.0,     60_636.0),
        'SRC_TO_DST_IAT_AVG'         : (0.0,     59_996.0),
        'SRC_TO_DST_IAT_STDDEV'      : (0.0,     29_325.0),
        'SRC_TO_DST_AVG_THROUGHPUT'  : (6.0,     14_654_532.0),
        'NUM_PKTS_UP_TO_128_BYTES'   : (0.0,     140_225.0),
    }

    def __post_init__(self):
        n = len(self.feature_names)
        self.forward_mask    = self._build_mask(self.FORWARD_SURR, n)
        self.perturbable_mask = self.forward_mask.copy()

    def _build_mask(self, feature_list, n):
        mask = np.zeros(n, dtype=bool)
        for feat in feature_list:
            idx = self._feat_idx(feat)
            if idx is not None:
                mask[idx] = True
        return mask

    def _feat_idx(self, name):
        matches = np.where(self.feature_names == name)[0]
        return int(matches[0]) if len(matches) > 0 else None

    def to_physical_space(self, X_scaled):
        return self.scaler.inverse_transform(X_scaled).astype(np.float64)

    def to_scaled_space(self, X_raw):
        return self.scaler.transform(X_raw).astype(np.float32)

    def apply_causal_graph(self, X_phys):
        """
        Saneamiento causal sobre las 30 features del surrogate.
        Solo recalcula las derivadas disponibles en NF-UQ.
        """
        def safe_div(num, den):
            return num / (den + 1e-8)

        i_in_bytes  = self._feat_idx('IN_BYTES')
        i_in_pkts   = self._feat_idx('IN_PKTS')
        i_out_bytes = self._feat_idx('OUT_BYTES')
        i_out_pkts  = self._feat_idx('OUT_PKTS')
        i_dur       = self._feat_idx('FLOW_DURATION_MILLISECONDS')

        # Clipping físico absoluto
        for feat, (lo, hi) in self.PHYSICAL_BOUNDS_SURR.items():
            idx = self._feat_idx(feat)
            if idx is not None:
                X_phys[:, idx] = np.clip(X_phys[:, idx], lo, hi)

        # Recalcular derivadas disponibles
        idx = self._feat_idx('TOTAL_BYTES')
        if idx is not None and i_in_bytes is not None and i_out_bytes is not None:
            X_phys[:, idx] = X_phys[:, i_in_bytes] + X_phys[:, i_out_bytes]

        idx = self._feat_idx('SRC_BYTES_PER_PKT')
        if idx is not None and i_in_bytes is not None and i_in_pkts is not None:
            X_phys[:, idx] = safe_div(X_phys[:, i_in_bytes], X_phys[:, i_in_pkts])

        idx = self._feat_idx('PKT_RATIO')
        if idx is not None and i_in_pkts is not None and i_out_pkts is not None:
            X_phys[:, idx] = safe_div(X_phys[:, i_in_pkts], X_phys[:, i_out_pkts])

        idx = self._feat_idx('IS_UNIDIRECTIONAL')
        if idx is not None and i_out_pkts is not None:
            X_phys[:, idx] = np.where(X_phys[:, i_out_pkts] == 0, 1.0, 0.0)

        return X_phys

    def summary(self):
        print(f"SurrogateDC | Features: {len(self.feature_names)} | "
              f"Forward: {self.forward_mask.sum()} | "
              f"Frozen: {(~self.forward_mask).sum()}")


# Instanciar DC del surrogate
# El scaler ya viene del pipeline de preprocesamiento de NF-UQ
surrogate_scaler = joblib.load(
    os.path.join(SURROGATE_MODELS_PATH, 'quantile_scaler.pkl') # <-- CAMBIO AQUÍ
)

dc_surrogate = SurrogateDomainConstraints(
    feature_names = feature_names_surrogate,
    scaler        = surrogate_scaler,
)
dc_surrogate.summary()

print(f"\n   Forward surrogate ({dc_surrogate.forward_mask.sum()} features):")
for i, name in enumerate(feature_names_surrogate):
    if dc_surrogate.forward_mask[i]:
        print(f"     [{i:>2}] {name}")

## **Wrapper del surrogate + mapeo de espacios**

In [ ]:
import os
import joblib
import numpy as np

"""
Wrapper del surrogate compatible con la interfaz de los ataques.
Mapeo bidireccional entre el espacio de 30 features (surrogate)
y el espacio de 66 features (víctima).
"""

print("[-] Cargando LightGBM Surrogate desde disco...")
surrogate_path = os.path.join(SURROGATE_MODELS_PATH, 'surrogate_lgbm.pkl')
surrogate_lgbm = joblib.load(surrogate_path)
print(f"   [✓] Surrogate cargado. Total de árboles: {surrogate_lgbm.booster_.num_trees()}\n")

class SurrogateWrapper:
    """
    Wrapper que expone predict() y predict_proba() del surrogate
    con la misma interfaz que lgbm_victim.
    LightGBM del surrogate trabaja en espacio raw (sin escalar).
    """
    def __init__(self, lgbm_model, dc_surr):
        self.model   = lgbm_model
        self.dc_surr = dc_surr

    def predict(self, X):
        # X llega en espacio escalado del surrogate → invertir antes de predecir
        X_raw = self.dc_surr.to_physical_space(X)
        return self.model.predict(X_raw)

    def predict_proba(self, X):
        X_raw = self.dc_surr.to_physical_space(X)
        return self.model.predict_proba(X_raw)

surrogate_wrapper = SurrogateWrapper(surrogate_lgbm, dc_surrogate)

# Mapeo de índices surrogate ↔ víctima
print("[-] Construyendo mapeo surrogate ↔ víctima...")

surrogate_to_victim = {}  # surr_idx → victim_idx
victim_to_surrogate = {}  # victim_idx → surr_idx

for surr_idx, feat in enumerate(feature_names_surrogate):
    victim_matches = np.where(dc.feature_names == feat)[0]
    if len(victim_matches) > 0:
        victim_idx = int(victim_matches[0])
        surrogate_to_victim[surr_idx] = victim_idx
        victim_to_surrogate[victim_idx] = surr_idx

n_mapeadas = len(surrogate_to_victim)
print(f"   [✓] Features mapeadas: {n_mapeadas}/{len(feature_names_surrogate)}")
print(f"   [✓] Features BUF_* no disponibles para el atacante: "
      f"{dc.immutable_mask.sum()} (se mantienen del original)")


def project_victim_to_surrogate(X_victim_raw):
    """
    Proyecta flujos del espacio víctima (66 raw) al espacio surrogate (30 raw).
    Solo extrae las features que el atacante conoce.
    """
    n = len(X_victim_raw)
    X_surr = np.zeros((n, len(feature_names_surrogate)), dtype=np.float32)
    for surr_idx, victim_idx in surrogate_to_victim.items():
        X_surr[:, surr_idx] = X_victim_raw[:, victim_idx]
    return X_surr


def expand_surrogate_to_victim(X_surr_adv_raw, X_victim_raw_original):
    """
    Expande adversariales del surrogate (30 raw) al espacio víctima (66 raw).

    Las features que el atacante modificó se aplican al víctima.
    Las BUF_* y features sintéticas se mantienen del original —
    el atacante no las conoce ni puede modificarlas.

    Retorna:
      X_expanded_raw : (n, 66) en espacio físico para LightGBM víctima
      X_expanded_sc  : (n, 66) en espacio escalado para ResNet víctima
    """
    X_expanded_raw = X_victim_raw_original.copy()

    # Aplicar solo las perturbaciones en features conocidas por el atacante
    for surr_idx, victim_idx in surrogate_to_victim.items():
        X_expanded_raw[:, victim_idx] = X_surr_adv_raw[:, surr_idx]

    # Aplicar causal graph del víctima para coherencia física completa
    X_expanded_raw = dc.apply_causal_graph(X_expanded_raw)

    # Escalar para ResNet
    X_expanded_sc = dc.to_scaled_space(X_expanded_raw)

    return X_expanded_raw, X_expanded_sc


# Proyectar flujos víctima al espacio surrogate
X_victim_in_surr_raw = project_victim_to_surrogate(X_victim_attacks_raw)
X_victim_in_surr_sc  = dc_surrogate.to_scaled_space(X_victim_in_surr_raw)

print(f"\n   [✓] Flujos víctima en espacio surrogate (raw): {X_victim_in_surr_raw.shape}")
print(f"   [✓] Flujos víctima en espacio surrogate (sc) : {X_victim_in_surr_sc.shape}")

# Validar que el surrogate detecta los ataques del víctima
y_pred_surr_orig = surrogate_lgbm.predict(X_victim_in_surr_raw)
detection_surr   = (y_pred_surr_orig != 0).mean()
print(f"\n   Detección surrogate sobre flujos BigFlow: {detection_surr*100:.1f}%")
print(f"   Detección víctima LightGBM              : {detection_lgbm*100:.1f}%")
print(f"   Detección víctima ResNet                : {detection_resnet*100:.1f}%")

# Solo atacar lo que el surrogate también detecta — experimento limpio
mask_detected_by_surr = y_pred_surr_orig != 0
n_validas = mask_detected_by_surr.sum()
print(f"\n   Muestras válidas (detectadas por ambos): {n_validas:,}/{len(y_victim_attacks):,}")
print(f"   Estas son las que usaremos para SPECTRA — ITF honesto")

## HIPÓTESIS:

- Surrogate (30 feat, sin buffer) → 19.3% detección

- Víctima   (66 feat, con buffer) → 98%+  detección

Diferencia: +78.7 puntos porcentuales

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

modelos  = ['Surrogate\n(30 feat, sin buffer)',
            'LightGBM víctima\n(66 feat, con buffer)',
            'ResNet víctima\n(66 feat, con buffer)']
valores  = [detection_surr*100, detection_lgbm*100, detection_resnet*100]
colores  = ['#BA7517', '#E24B4A', '#1D9E75']

bars = ax.bar(modelos, valores, color=colores, width=0.5, edgecolor='none')
ax.set_ylabel('Tasa de detección de ataques (%)', fontweight='bold')
ax.set_title('Impacto del IP Behavior Buffer en la detección\n'
             '(mismo conjunto de 2500 flujos de ataque BigFlow-NIDS)',
             fontweight='bold', pad=12)
ax.set_ylim(0, 115)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Umbral 50%')

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

ax.grid(axis='y', linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

# **SPECTRA: LEAF + THORN dual sobre surrogate**

In [ ]:
"""
SPECTRA — Ataque dual LEAF + THORN sobre el surrogate.
El atacante usa su propio modelo con su propio DC para generar
adversariales físicamente válidos en su espacio de 30 features.
Luego los expande al espacio de 66 features del víctima.
"""
from src.attacks.leaf import LEAFAttack
from src.attacks.thorn import THORNAttack
import time
import warnings

# --- SILENCIADOR DE WARNINGS ---
# Ignoramos el spam de "X does not have valid feature names..."
warnings.filterwarnings("ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'resultados_spectra')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("="*80)
print("SPECTRA — Transferibilidad Grey-Box: LEAF + THORN sobre Surrogate")
print("="*80)
print(f"   Surrogate: LightGBM NF-UQ-NIDS-V2 ({len(feature_names_surrogate)} features)")
print(f"   Víctima  : LightGBM + ResNet BigFlow-NIDS (66 features)")
print(f"   DC       : SurrogateDC con restricciones físicas del protocolo")
print()

# 1. Filtramos PRIMERO: Solo atacar lo que el surrogate también detecta (ITF honesto)
mask_detected_by_surr = y_pred_surr_orig != 0
n_validas = mask_detected_by_surr.sum()
print(f"[*] Muestras válidas (detectadas por el clon): {n_validas:,}/{len(y_victim_attacks):,}")
print(f"    Estas son las que usaremos para SPECTRA.")

# Subconjunto válido
X_surr_sc_valid  = X_victim_in_surr_sc[mask_detected_by_surr]
X_surr_raw_valid = X_victim_in_surr_raw[mask_detected_by_surr]
X_vic_raw_valid  = X_victim_attacks_raw[mask_detected_by_surr]
y_valid          = y_victim_attacks[mask_detected_by_surr]

# 2. Configuración de Ataques
epsilons_spectra  = [0.05, 0.15, 0.30, 0.50, 1.0, 2.0]
surrogate_booster = surrogate_lgbm.booster_
resultados_spectra = []

for eps in epsilons_spectra:
    row = {"Epsilon": eps}

    # ── LEAF sobre surrogate ──────────────────────────────────────────────
    atk_leaf_surr = LEAFAttack(
        constraints = dc_surrogate,
        model_trees = surrogate_booster,
        epsilon     = eps,
        top_k       = 5,
        mode        = 'targeted',
        verbose     = False,
    )

    t0 = time.time()
    res_leaf = atk_leaf_surr.run(X_surr_sc_valid, y_valid, surrogate_wrapper)
    t_leaf   = time.time() - t0

    # Expandir al espacio víctima y evaluar
    X_leaf_adv_raw = dc_surrogate.to_physical_space(res_leaf.X_adv)
    X_leaf_vic_raw, X_leaf_vic_sc = expand_surrogate_to_victim(
        X_leaf_adv_raw, X_vic_raw_valid
    )

    asr_leaf_surr  = res_leaf.asr * 100
    asr_leaf_lgbm  = (lgbm_victim.predict(X_leaf_vic_raw) == 0).mean() * 100
    asr_leaf_resnet = (resnet_victim.predict(X_leaf_vic_sc) == 0).mean() * 100

    # ITF = Transferibilidad. Si el clon no evadió nada, el ITF es 0.
    itf_leaf_lgbm  = asr_leaf_lgbm / asr_leaf_surr if asr_leaf_surr > 0.5 else 0.0
    itf_leaf_resnet = asr_leaf_resnet / asr_leaf_surr if asr_leaf_surr > 0.5 else 0.0

    l0_leaf = (np.abs(res_leaf.X_adv - X_surr_sc_valid) > 1e-4).sum(axis=1).mean()
    l2_leaf = np.linalg.norm(res_leaf.X_adv - X_surr_sc_valid, axis=1).mean()

    row.update({
        "LEAF ASR Surrogate (%)": round(asr_leaf_surr, 2),
        "LEAF ASR LightGBM (%)": round(asr_leaf_lgbm, 2),
        "LEAF ASR ResNet (%)": round(asr_leaf_resnet, 2),
        "LEAF ITF LightGBM": round(itf_leaf_lgbm, 3),
        "LEAF ITF ResNet": round(itf_leaf_resnet, 3),
        "LEAF L0": round(l0_leaf, 2),
        "LEAF L2": round(l2_leaf, 4),
        "LEAF T(s)": round(t_leaf, 1),
    })

    # ── THORN sobre surrogate ─────────────────────────────────────────────
    atk_thorn_surr = THORNAttack(
        constraints = dc_surrogate,
        model_trees = surrogate_booster,
        epsilon     = eps,
        num_trees   = 50, # <--- ¡VITAL! Evita la explosión combinatoria
        verbose     = False,
    )

    t0 = time.time()
    res_thorn = atk_thorn_surr.run(X_surr_sc_valid, y_valid, surrogate_wrapper)
    t_thorn   = time.time() - t0

    X_thorn_adv_raw = dc_surrogate.to_physical_space(res_thorn.X_adv)
    X_thorn_vic_raw, X_thorn_vic_sc = expand_surrogate_to_victim(
        X_thorn_adv_raw, X_vic_raw_valid
    )

    asr_thorn_surr   = res_thorn.asr * 100
    asr_thorn_lgbm   = (lgbm_victim.predict(X_thorn_vic_raw) == 0).mean() * 100
    asr_thorn_resnet = (resnet_victim.predict(X_thorn_vic_sc) == 0).mean() * 100

    itf_thorn_lgbm   = asr_thorn_lgbm / asr_thorn_surr if asr_thorn_surr > 0.5 else 0.0
    itf_thorn_resnet = asr_thorn_resnet / asr_thorn_surr if asr_thorn_surr > 0.5 else 0.0

    l0_thorn = (np.abs(res_thorn.X_adv - X_surr_sc_valid) > 1e-4).sum(axis=1).mean()
    l2_thorn = np.linalg.norm(res_thorn.X_adv - X_surr_sc_valid, axis=1).mean()

    row.update({
        "THORN ASR Surrogate (%)": round(asr_thorn_surr, 2),
        "THORN ASR LightGBM (%)": round(asr_thorn_lgbm, 2),
        "THORN ASR ResNet (%)": round(asr_thorn_resnet, 2),
        "THORN ITF LightGBM": round(itf_thorn_lgbm, 3),
        "THORN ITF ResNet": round(itf_thorn_resnet, 3),
        "THORN L0": round(l0_thorn, 2),
        "THORN L2": round(l2_thorn, 4),
        "THORN T(s)": round(t_thorn, 1),
    })

    resultados_spectra.append(row)

    print(f"[*] ε={eps:.2f} | "
          f"LEAF surr={asr_leaf_surr:.1f}% lgbm={asr_leaf_lgbm:.1f}% resnet={asr_leaf_resnet:.1f}% | "
          f"THORN surr={asr_thorn_surr:.1f}% lgbm={asr_thorn_lgbm:.1f}% resnet={asr_thorn_resnet:.1f}%")

df_spectra = pd.DataFrame(resultados_spectra)
joblib.dump(df_spectra, os.path.join(CHECKPOINT_DIR, "resultados_spectra.pkl"))
print(f"\n[✔] Resultados SPECTRA guardados en {CHECKPOINT_DIR}")

# Separamos la visualización final para que sea más clara
df_display = df_spectra[['Epsilon', 'LEAF ASR Surrogate (%)', 'LEAF ITF LightGBM', 'LEAF ITF ResNet',
                         'THORN ASR Surrogate (%)', 'THORN ITF LightGBM', 'THORN ITF ResNet']]
display(df_display.style.background_gradient(cmap='YlOrRd', subset=['LEAF ITF LightGBM', 'THORN ITF LightGBM']))

In [ ]:
import os
import joblib
import pandas as pd
from IPython.display import display

# 1. Definimos la ruta que faltaba (usando PROJECT_ROOT que ya tenías)
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'resultados_spectra')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2. Rescatamos los datos que siguen vivos en la RAM
df_spectra = pd.DataFrame(resultados_spectra)

# 3. Guardamos a disco
joblib.dump(df_spectra, os.path.join(CHECKPOINT_DIR, "resultados_spectra.pkl"))
print(f"[✔] ¡Datos rescatados de la RAM! Resultados SPECTRA guardados en {CHECKPOINT_DIR}")

# 4. Mostramos la tabla final
df_display = df_spectra[['Epsilon', 'LEAF ASR Surrogate (%)', 'LEAF ITF LightGBM', 'LEAF ITF ResNet',
                         'THORN ASR Surrogate (%)', 'THORN ITF LightGBM', 'THORN ITF ResNet']]
display(df_display.style.background_gradient(cmap='YlOrRd', subset=['LEAF ITF LightGBM', 'THORN ITF LightGBM']))

# **Búsqueda del exploit óptimo por clase**
- Contribución original y solución realista para el TFG

# **PROPUESTA DE SIMULACIÓN DE ATAQUE APT SOFISTICADO PARA SPECTRA**

In [ ]:
"""
SPECTRA FORENSICS: Búsqueda Dual de Escalada (LEAF + THORN)
Simula a un atacante real (APT): busca la evasión absoluta (Caja Gris -> Caja Negra)
probando tanto ataques de ruta (THORN) como ataques de hojas (LEAF),
y se queda con el payload que genere la menor huella L2 posible.
"""
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from src.attacks.leaf import LEAFAttack
from src.attacks.thorn import THORNAttack

print("="*85)
print("ANÁLISIS FORENSE AVANZADO — Búsqueda Dual de Exploit Mínimo Viable")
print("="*85)

CLASS_NAMES = {
    0: 'Benign', 1: 'DoS', 2: 'DDoS', 3: 'Web/Injection',
    4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'
}

# 1. Parrilla de presupuestos de ataque (del bisturí al martillo)
eps_grid = [0.01, 0.05, 0.10, 0.20, 0.35, 0.50, 1.0, 2.0]

# 2. Registro Maestro de Exploits (inicializado al peor caso)
n_samples = len(X_surr_sc_valid)
best_exploits_global = {
    i: {'evaded': False, 'eps': np.inf, 'l2': np.inf, 'l0': np.inf,
        'attack_used': 'None', 'X_adv_surr': None,
        'X_adv_vic_raw': None, 'X_adv_vic_sc': None}
    for i in range(n_samples)
}

print(f"[*] Iniciando explotación de sigilo sobre {n_samples} flujos válidos...\n")

# 3. BUCLE DE ESCALADA DUAL
for eps in tqdm(eps_grid, desc="💀 SPECTRA Dual Search", unit="eps", colour="red"):

    # --- A. ATAQUE LEAF (La Ganzúa) ---
    atk_leaf = LEAFAttack(
        constraints = dc_surrogate, model_trees = surrogate_booster,
        epsilon = eps, top_k = 5, mode = 'targeted', verbose = False
    )
    res_leaf = atk_leaf.run(X_surr_sc_valid, y_valid, surrogate_wrapper)
    X_leaf_vic_raw, X_leaf_vic_sc = expand_surrogate_to_victim(
        dc_surrogate.to_physical_space(res_leaf.X_adv), X_vic_raw_valid
    )
    # Predicción Víctima
    evaded_leaf = (lgbm_victim.predict(X_leaf_vic_raw) == 0) & (resnet_victim.predict(X_leaf_vic_sc) == 0)
    l2_leaf = np.linalg.norm(res_leaf.X_adv - X_surr_sc_valid, axis=1)
    l0_leaf = (np.abs(res_leaf.X_adv - X_surr_sc_valid) > 1e-4).sum(axis=1)

    # --- B. ATAQUE THORN (La Palanca) ---
    atk_thorn = THORNAttack(
        constraints = dc_surrogate, model_trees = surrogate_booster,
        epsilon = eps, num_trees = 50, verbose = False
    )
    res_thorn = atk_thorn.run(X_surr_sc_valid, y_valid, surrogate_wrapper)
    X_thorn_vic_raw, X_thorn_vic_sc = expand_surrogate_to_victim(
        dc_surrogate.to_physical_space(res_thorn.X_adv), X_vic_raw_valid
    )
    # Predicción Víctima
    evaded_thorn = (lgbm_victim.predict(X_thorn_vic_raw) == 0) & (resnet_victim.predict(X_thorn_vic_sc) == 0)
    l2_thorn = np.linalg.norm(res_thorn.X_adv - X_surr_sc_valid, axis=1)
    l0_thorn = (np.abs(res_thorn.X_adv - X_surr_sc_valid) > 1e-4).sum(axis=1)

    # --- C. COMPETICIÓN Y ACTUALIZACIÓN DEL REGISTRO ---
    for i in range(n_samples):
        # Si LEAF evade y mejora la huella L2 histórica
        if evaded_leaf[i] and l2_leaf[i] < best_exploits_global[i]['l2']:
            best_exploits_global[i] = {
                'evaded': True, 'eps': eps, 'l2': l2_leaf[i], 'l0': l0_leaf[i],
                'attack_used': 'LEAF', 'X_adv_surr': res_leaf.X_adv[i],
                'X_adv_vic_raw': X_leaf_vic_raw[i], 'X_adv_vic_sc': X_leaf_vic_sc[i]
            }
        # Si THORN evade y es mejor (incluso mejor que el LEAF que acabamos de calcular)
        if evaded_thorn[i] and l2_thorn[i] < best_exploits_global[i]['l2']:
            best_exploits_global[i] = {
                'evaded': True, 'eps': eps, 'l2': l2_thorn[i], 'l0': l0_thorn[i],
                'attack_used': 'THORN', 'X_adv_surr': res_thorn.X_adv[i],
                'X_adv_vic_raw': X_thorn_vic_raw[i], 'X_adv_vic_sc': X_thorn_vic_sc[i]
            }

# 4. EXTRACCIÓN DE RESULTADOS POR CLASE
print(f"\n{'─'*95}")
print(f"  {'Clase':<16} {'N válidas':>10} {'N evadidas':>10} {'ASR (%)':>8} "
      f"{'L2 mín':>8} {'L0 mín':>6}  {'Mejor Motor':<12} {'Eps usado':>8}")
print(f"{'─'*95}")

exploits_por_clase = {}
total_evadidas = sum(1 for v in best_exploits_global.values() if v['evaded'])

for cls_id, cls_name in CLASS_NAMES.items():
    if cls_id == 0: continue

    idx_clase = np.where(y_valid == cls_id)[0]
    n_validas = len(idx_clase)
    if n_validas == 0: continue

    evadidos_clase = [i for i in idx_clase if best_exploits_global[i]['evaded']]
    n_evadidas = len(evadidos_clase)
    asr_cls = (n_evadidas / n_validas) * 100

    if n_evadidas > 0:
        # Extraemos el "Exploit Dorado" (el de menor L2 de toda la clase)
        mejor_idx = min(evadidos_clase, key=lambda x: best_exploits_global[x]['l2'])
        mejor_data = best_exploits_global[mejor_idx]

        exploits_por_clase[cls_id] = {
            'cls_name'    : cls_name,
            'idx_global'  : mejor_idx,
            'X_orig_surr' : X_surr_sc_valid[mejor_idx],
            'X_adv_surr'  : mejor_data['X_adv_surr'],
            'X_orig_vic'  : X_vic_raw_valid[mejor_idx],
            'X_adv_vic'   : mejor_data['X_adv_vic_raw'],
            'l2_min'      : mejor_data['l2'],
            'l0_min'      : mejor_data['l0'],
            'eps_optimo'  : mejor_data['eps'],
            'asr_cls'     : asr_cls,
        }

        print(f"  {cls_name:<16} {n_validas:>10,} {n_evadidas:>10,} "
              f"{asr_cls:>8.1f}% {mejor_data['l2']:>8.4f} {mejor_data['l0']:>6}  "
              f"{mejor_data['attack_used']:<12} {mejor_data['eps']:>8.2f}")
    else:
        print(f"  {cls_name:<16} {n_validas:>10,} {'0':>10} "
              f"{'0.0%':>9} {'—':>8} {'—':>6}  {'—':<12} {'—':>8}")

print(f"{'─'*95}")
print(f"  Resumen Global: {total_evadidas}/{n_samples} flujos evadieron ambos modelos (ASR Total Dual: {(total_evadidas/n_samples)*100:.1f}%)")

# **Análisis semántico del exploit por clase**

In [ ]:
"""
Para cada exploit mínimo viable, mostrar exactamente qué features
modificó el ataque y traducirlo a lenguaje táctico para un analista.
"""

print("="*80)
print("ANÁLISIS SEMÁNTICO — Qué modifica el atacante en cada exploit")
print("="*80)

# Descripciones tácticas de las features Forward
FEATURE_SEMANTICS = {
    'IN_BYTES'                   : 'Bytes enviados por el atacante',
    'IN_PKTS'                    : 'Paquetes enviados por el atacante',
    'MIN_TTL'                    : 'TTL mínimo (distancia de red aparente)',
    'LONGEST_FLOW_PKT'           : 'Tamaño del paquete más grande',
    'SHORTEST_FLOW_PKT'          : 'Tamaño del paquete más pequeño',
    'MIN_IP_PKT_LEN'             : 'Longitud mínima de paquete IP',
    'MAX_IP_PKT_LEN'             : 'Longitud máxima de paquete IP',
    'SRC_TO_DST_AVG_THROUGHPUT'  : 'Velocidad media de envío (bytes/s)',
    'TCP_WIN_MAX_IN'             : 'Ventana TCP máxima anunciada',
    'SRC_TO_DST_IAT_AVG'         : 'Intervalo medio entre paquetes (ms)',
    'SRC_TO_DST_IAT_STDDEV'      : 'Variabilidad del timing (ms)',
    'FLOW_DURATION_MILLISECONDS' : 'Duración total del flujo (ms)',
    'NUM_PKTS_UP_TO_128_BYTES'   : 'Paquetes pequeños (≤128 bytes)',
    'NUM_PKTS_512_TO_1024_BYTES' : 'Paquetes medianos (512-1024 bytes)',
    'NUM_PKTS_1024_TO_1514_BYTES': 'Paquetes grandes (1024-1514 bytes)',
}

for cls_id, exploit in exploits_por_clase.items():
    cls_name = exploit['cls_name']
    X_orig   = exploit['X_orig_surr']
    X_adv    = exploit['X_adv_surr']

    # Calcular delta en espacio físico para interpretabilidad
    X_orig_phys = dc_surrogate.to_physical_space(X_orig[np.newaxis])[0]
    X_adv_phys  = dc_surrogate.to_physical_space(X_adv[np.newaxis])[0]
    delta_phys  = X_adv_phys - X_orig_phys

    # Features modificadas (L0)
    modified_mask = np.abs(X_adv - X_orig) > 1e-4
    modified_idx  = np.where(modified_mask)[0]

    print(f"\n{'═'*70}")
    print(f"  CLASE: {cls_name} | L2={exploit['l2_min']:.4f} | "
          f"L0={exploit['l0_min']} features | ASR clase={exploit['asr_cls']:.1f}%")
    print(f"{'═'*70}")
    print(f"\n  Features modificadas por el ataque:")
    print(f"  {'Feature':<35} {'Antes':>12} {'Después':>12} {'Δ%':>8}")
    print(f"  {'─'*70}")

    cambios_significativos = []
    for idx in modified_idx:
        if idx >= len(feature_names_surrogate):
            continue
        feat_name = feature_names_surrogate[idx]
        v_orig    = X_orig_phys[idx]
        v_adv     = X_adv_phys[idx]
        delta_pct = ((v_adv - v_orig) / (abs(v_orig) + 1e-8)) * 100

        print(f"  {feat_name:<35} {v_orig:>12.2f} {v_adv:>12.2f} {delta_pct:>+8.1f}%")
        cambios_significativos.append((feat_name, v_orig, v_adv, delta_pct))

    # Recomendación táctica automática
    print(f"\n  RECOMENDACIÓN TÁCTICA para evadir detección de {cls_name}:")
    for feat_name, v_orig, v_adv, delta_pct in cambios_significativos:
        semantica = FEATURE_SEMANTICS.get(feat_name, feat_name)
        direccion = "aumentar" if delta_pct > 0 else "reducir"
        print(f"    → {direccion.upper()} '{semantica}' un {abs(delta_pct):.1f}%"
              f" ({v_orig:.1f} → {v_adv:.1f})")

## HIPÓTESIS Y JUSTIFICACIÓN

Para evadir la detección de DoS, el ataque recomienda reducir los paquetes y bytes en un 70%. Pero es cierto que un DoS se detecta por volumen. Si el atacante reduce el volumen, el NIDS no lo ve... pero deja de ser un DoS efectivo.

En cambio, para el Malware, el ataque recomienda aumentar la velocidad y tamaño. Como el malware suele ser un tráfico "goteo" (C2 beacons), inyectar ruido o hacerlo un poco más rápido descoloca al modelo sin romper el payload malicioso.

# **Gráficas de transferibilidad y sigilo**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

epsilons_plot = df_spectra['Epsilon'].values
colores = {
    'LEAF_lgbm'   : '#6A3D9A', 'THORN_lgbm'  : '#FF7F00',
    'LEAF_resnet' : '#A8DDB5', 'THORN_resnet' : '#FDD0A2',
    'LEAF_surr'   : '#BCBDDC', 'THORN_surr'   : '#FDAE6B',
}

# ── Panel 1: ASR sobre surrogate ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epsilons_plot, df_spectra['LEAF ASR Surrogate (%)'],
         'o--', color=colores['LEAF_surr'], lw=2, ms=7, label='LEAF Surrogate')
ax1.plot(epsilons_plot, df_spectra['THORN ASR Surrogate (%)'],
         's-',  color=colores['THORN_surr'], lw=2, ms=7, label='THORN Surrogate')
ax1.set_title('ASR sobre Surrogate\n(Modelo del Atacante)', fontweight='bold')
ax1.set_xlabel('Epsilon'); ax1.set_ylabel('ASR (%)')
ax1.set_xticks(epsilons_plot); ax1.grid(True, ls=':', alpha=0.6)
ax1.legend(fontsize=9)

# ── Panel 2: ASR transferido a LightGBM víctima ───────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epsilons_plot, df_spectra['LEAF ASR LightGBM (%)'],
         'o--', color=colores['LEAF_lgbm'], lw=2, ms=7, label='LEAF → LightGBM')
ax2.plot(epsilons_plot, df_spectra['THORN ASR LightGBM (%)'],
         's-',  color=colores['THORN_lgbm'], lw=2, ms=7, label='THORN → LightGBM')
ax2.set_title('ASR Transferido\nLightGBM Víctima', fontweight='bold')
ax2.set_xlabel('Epsilon'); ax2.set_ylabel('ASR (%)')
ax2.set_xticks(epsilons_plot); ax2.grid(True, ls=':', alpha=0.6)
ax2.legend(fontsize=9)

# ── Panel 3: ASR transferido a ResNet víctima ─────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epsilons_plot, df_spectra['LEAF ASR ResNet (%)'],
         'o--', color=colores['LEAF_resnet'], lw=2, ms=7, label='LEAF → ResNet')
ax3.plot(epsilons_plot, df_spectra['THORN ASR ResNet (%)'],
         's-',  color=colores['THORN_resnet'], lw=2, ms=7, label='THORN → ResNet')
ax3.set_title('ASR Transferido\nResNet Víctima', fontweight='bold')
ax3.set_xlabel('Epsilon'); ax3.set_ylabel('ASR (%)')
ax3.set_xticks(epsilons_plot); ax3.grid(True, ls=':', alpha=0.6)
ax3.legend(fontsize=9)

# ── Panel 4: ITF (Índice de Transferibilidad de Frontera) ─────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(epsilons_plot, df_spectra['LEAF ITF LightGBM'],
         'o--', color=colores['LEAF_lgbm'], lw=2, ms=7, label='LEAF ITF LightGBM')
ax4.plot(epsilons_plot, df_spectra['THORN ITF LightGBM'],
         's-',  color=colores['THORN_lgbm'], lw=2, ms=7, label='THORN ITF LightGBM')
ax4.axhline(0.7, color='red', ls='--', lw=1.5, alpha=0.7, label='Umbral crítico (0.7)')
ax4.axhline(0.4, color='orange', ls='--', lw=1.5, alpha=0.7, label='Umbral medio (0.4)')
ax4.set_title('Índice de Transferibilidad\nde Frontera (ITF)', fontweight='bold')
ax4.set_xlabel('Epsilon'); ax4.set_ylabel('ITF')
ax4.set_xticks(epsilons_plot); ax4.grid(True, ls=':', alpha=0.6)
ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8)

# ── Panel 5: L0 vs ASR (curva de eficiencia) ──────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.scatter(df_spectra['LEAF L0'],  df_spectra['LEAF ASR LightGBM (%)'],
            c=epsilons_plot, cmap='Blues', s=100, label='LEAF', marker='o',
            edgecolors='black', linewidths=0.5)
sc = ax5.scatter(df_spectra['THORN L0'], df_spectra['THORN ASR LightGBM (%)'],
                  c=epsilons_plot, cmap='Oranges', s=100, label='THORN', marker='s',
                  edgecolors='black', linewidths=0.5)
ax5.set_title('Curva de Eficiencia\nL0 (sigilo) vs ASR', fontweight='bold')
ax5.set_xlabel('L0 — Features modificadas (↓ más sigiloso)')
ax5.set_ylabel('ASR LightGBM (%)')
ax5.grid(True, ls=':', alpha=0.6); ax5.legend(fontsize=9)

# ── Panel 6: Heatmap de ASR por clase ─────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
clases_con_exploit = list(exploits_por_clase.keys())
asr_por_clase = [exploits_por_clase[c]['asr_cls'] for c in clases_con_exploit]
nombres_clase = [exploits_por_clase[c]['cls_name'] for c in clases_con_exploit]
l2_por_clase  = [exploits_por_clase[c]['l2_min'] for c in clases_con_exploit]

bars = ax6.barh(nombres_clase, asr_por_clase,
                color=plt.cm.RdYlGn([a/100 for a in asr_por_clase]),
                edgecolor='black', linewidth=0.5)
for bar, l2 in zip(bars, l2_por_clase):
    ax6.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'L2={l2:.3f}', va='center', fontsize=8)
ax6.set_title('ASR por Clase de Ataque\n(exploit mínimo viable)', fontweight='bold')
ax6.set_xlabel('ASR (%)'); ax6.grid(True, ls=':', alpha=0.6, axis='x')

fig.suptitle(
    'SPECTRA — Transferibilidad Grey-Box: LEAF + THORN sobre Surrogate NF-UQ-NIDS-V2\n'
    'El atacante usa su propio modelo sin acceso al IDS víctima',
    fontsize=14, fontweight='bold', y=1.02
)
plt.savefig(
    os.path.join(CHECKPOINT_DIR, 'spectra_transferibilidad.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()
print("[✔] Gráfica guardada")

## 5. Análisis del rol defensivo del IP Behavior Buffer

Si el ITF es bajo, el buffer es la razón principal.
Demostramos esto comparando la predicción del víctima con y sin las features BUF_*.

In [ ]:
"""
ESTUDIO DE ABLACIÓN DEL IP BEHAVIOR BUFFER
Comparamos el daño de SPECTRA (THORN eps=2.0) contra el IDS real
y contra un IDS "ciego" al que le borramos el historial temporal de la IP.
"""
import numpy as np
import pandas as pd
from src.attacks.thorn import THORNAttack

print("="*80)
print("DEMOSTRACIÓN DE DEFENSA ACTIVA — Ablación del IP Buffer")
print("="*80)

# 1. Generamos el ataque de máxima potencia
eps_max = 2.0
atk_ablation = THORNAttack(
    constraints = dc_surrogate,
    model_trees = surrogate_booster,
    epsilon     = eps_max,
    num_trees   = 50,
    verbose     = False,
)

print(f"[-] Generando ataque THORN masivo (eps={eps_max}) sobre el Surrogate...")
res_abl = atk_ablation.run(X_surr_sc_valid, y_valid, surrogate_wrapper)

# 2. Expandimos al espacio físico de la víctima
X_adv_vic_raw_original, X_adv_vic_sc_original = expand_surrogate_to_victim(
    dc_surrogate.to_physical_space(res_abl.X_adv), X_vic_raw_valid
)

# 3. CREAMOS LA REALIDAD ALTERNATIVA (Sin IP Buffer)
print("[-] Borrando el historial del IP Buffer (Simulando IDS sin estado temporal)...")
# Obtenemos los índices de las features BUF_* en el espacio raw (66 features)
buf_mask = np.array(['BUF_' in name for name in dc.feature_names])
buf_indices = np.where(buf_mask)[0]

X_adv_vic_raw_no_buf = X_adv_vic_raw_original.copy()
# Ponemos a 0 el historial (como si fuera el primer paquete de una IP desconocida)
X_adv_vic_raw_no_buf[:, buf_indices] = 0.0

# Escalamos esta nueva realidad para la ResNet
X_adv_vic_sc_no_buf = dc.to_scaled_space(X_adv_vic_raw_no_buf)

# 4. PREDICCIONES CARA A CARA
print("[-] Evaluando impacto...\n")

# --- CON BUFFER (Realidad) ---
y_pred_lgbm_con = lgbm_victim.predict(X_adv_vic_raw_original)
y_pred_resnet_con = resnet_victim.predict(X_adv_vic_sc_original)
evasion_lgbm_con = (y_pred_lgbm_con == 0).mean() * 100
evasion_resnet_con = (y_pred_resnet_con == 0).mean() * 100

# --- SIN BUFFER (Ablación) ---
y_pred_lgbm_sin = lgbm_victim.predict(X_adv_vic_raw_no_buf)
y_pred_resnet_sin = resnet_victim.predict(X_adv_vic_sc_no_buf)
evasion_lgbm_sin = (y_pred_lgbm_sin == 0).mean() * 100
evasion_resnet_sin = (y_pred_resnet_sin == 0).mean() * 100

# 5. INFORME DE DEFENSA
print(f"{'─'*70}")
print(f"  {'Escenario':<25} {'Evasión LightGBM':>18} {'Evasión ResNet':>18}")
print(f"{'─'*70}")
print(f"  A) Con IP Buffer (Real)    {evasion_lgbm_con:>17.1f}% {evasion_resnet_con:>17.1f}%")
print(f"  B) Sin IP Buffer (Ciego)   {evasion_lgbm_sin:>17.1f}% {evasion_resnet_sin:>17.1f}%")
print(f"{'─'*70}")

delta_lgbm = evasion_lgbm_sin - evasion_lgbm_con
delta_resnet = evasion_resnet_sin - evasion_resnet_con

print(f"  [!] ATAQUES BLOQUEADOS POR EL BUFFER (Delta):")
print(f"      → LightGBM : Salvó al sistema de un {delta_lgbm:+.1f}% de ataques.")
print(f"      → ResNet   : Salvó al sistema de un {delta_resnet:+.1f}% de ataques.")

# Análisis detallado por clase (LightGBM)
print(f"\n  [!] IMPACTO POR CLASE EN LIGHTGBM (Con Buffer vs Sin Buffer):")
for cls_id, cls_name in CLASS_NAMES.items():
    if cls_id == 0: continue
    mask_cls = y_valid == cls_id
    if mask_cls.sum() == 0: continue

    ev_con = (y_pred_lgbm_con[mask_cls] == 0).mean() * 100
    ev_sin = (y_pred_lgbm_sin[mask_cls] == 0).mean() * 100
    print(f"      {cls_name:<15} : {ev_con:>5.1f}%  →  {ev_sin:>5.1f}%  (Δ {ev_sin-ev_con:>+5.1f}%)")

-------------------------

## 6. Visualización de resultados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Configuramos colores corporativos/académicos
COLOR_SURR = '#378ADD'  # Azul
COLOR_LGBM = '#E24B4A'  # Rojo
COLOR_RES  = '#1D9E75'  # Verde oscuro
COLOR_SIN_BUF = '#8C8C8C' # Gris para el "Sin Buffer"

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ==============================================================================
# Panel 1 — El Espejismo del Atacante (ASR sobre Surrogate vs Realidad)
# ==============================================================================
modelos = ['Clon Atacante\n(THORN eps=2.0)', 'Víctima\n(LightGBM)', 'Víctima\n(ResNet)']
asrs    = [res_abl.asr * 100, evasion_lgbm_con, evasion_resnet_con]
colores = [COLOR_SURR, COLOR_LGBM, COLOR_RES]

axes[0].bar(modelos, asrs, color=colores, width=0.5, edgecolor='black', linewidth=1)
axes[0].set_ylabel('Evasión (ASR %)', fontweight='bold')
axes[0].set_title('El Espejismo del Atacante\n(Ataque Caja Gris)', fontweight='bold', pad=15)
axes[0].set_ylim(0, max(asrs) * 1.2 + 5)
for i, v in enumerate(asrs):
    axes[0].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# ==============================================================================
# Panel 2 — Rol Defensivo del IP Buffer (Ablación Global)
# ==============================================================================
categorias = ['Con IP Buffer\n(Tu Arquitectura)', 'Sin IP Buffer\n(NIDS Tradicional)']
vals_resnet = [evasion_resnet_con, evasion_resnet_sin]
vals_lgbm   = [evasion_lgbm_con, evasion_lgbm_sin]

x = np.arange(len(categorias))
w = 0.3
axes[1].bar(x - w/2, vals_resnet, w, label='ResNet', color=COLOR_RES, edgecolor='black', linewidth=1)
axes[1].bar(x + w/2, vals_lgbm,   w, label='LightGBM', color=COLOR_LGBM, edgecolor='black', linewidth=1)
axes[1].set_ylabel('Ataques que lograron evadir (%)', fontweight='bold')
axes[1].set_title('Estudio de Ablación:\nEl IP Buffer como Escudo Activo', fontweight='bold', pad=15)
axes[1].set_xticks(x)
axes[1].set_xticklabels(categorias, fontweight='bold')

# Anotaciones de la degradación
axes[1].annotate(f'Δ +{evasion_resnet_sin - evasion_resnet_con:.1f}%', xy=(1 - w/2, evasion_resnet_sin + 2), ha='center', color=COLOR_RES, fontweight='bold')
axes[1].annotate(f'Δ +{evasion_lgbm_sin - evasion_lgbm_con:.1f}%', xy=(1 + w/2, evasion_lgbm_sin + 2), ha='center', color=COLOR_LGBM, fontweight='bold')

axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

# ==============================================================================
# Panel 3 — Colapso del NIDS por Clases (LightGBM)
# ==============================================================================
# Extraemos dinámicamente las métricas de la celda anterior
clases_labels = []
ev_con_list = []
ev_sin_list = []

for cls_id, cls_name in CLASS_NAMES.items():
    if cls_id == 0: continue
    mask_cls = y_valid == cls_id
    if mask_cls.sum() == 0: continue

    clases_labels.append(cls_name)
    ev_con_list.append((y_pred_lgbm_con[mask_cls] == 0).mean() * 100)
    ev_sin_list.append((y_pred_lgbm_sin[mask_cls] == 0).mean() * 100)

y_pos = np.arange(len(clases_labels))
h = 0.35

# Invertimos el orden para que se lea mejor de arriba a abajo
axes[2].barh(y_pos + h/2, ev_sin_list, h, label='Sin IP Buffer (Ciego)', color=COLOR_SIN_BUF, edgecolor='black', linewidth=1)
axes[2].barh(y_pos - h/2, ev_con_list, h, label='Con IP Buffer (Real)', color=COLOR_LGBM, edgecolor='black', linewidth=1)

axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(clases_labels, fontsize=10, fontweight='bold')
axes[2].set_xlabel('Evasión Efectiva (%)', fontweight='bold')
axes[2].set_title('Colapso de Seguridad por Tipo de Ataque\n(LightGBM)', fontweight='bold', pad=15)
axes[2].legend()
axes[2].grid(axis='x', linestyle='--', alpha=0.5)
axes[2].set_xlim(0, 105)

# Título General
plt.suptitle('SPECTRA — Evaluación de Robustez y Defensa Activa (IP Behavior Buffer)',
             fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()

# Guardar figura
plt.savefig(os.path.join(CHECKPOINT_DIR, 'spectra_ablacion_defensa.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"[✓] Figura de Ablación guardada en {CHECKPOINT_DIR}")